# EDA — APT29 / Mordor (siem_dataset)

**Phase 1 du PLAN.md** — Comprendre les données interactivement AVANT de modéliser.

## Questions auxquelles ce notebook répond

1. Combien d'events par jour, par channel, par EventID ?
2. Quels hôtes sont les plus actifs ?
3. Quelle est la timeline (frise temporelle) des events ?
4. Quelles commandes PowerShell suspectes apparaissent ?
5. **Combien d'events seraient étiquetés malveillants avec un prototype MITRE ?**
6. Quelles règles de nettoyage appliquer ?

## Stratégie

Le volume (2 GB JSON) ne tient pas confortablement en RAM en DataFrame. On va :
1. Streamer ligne par ligne les 2 ZIP
2. Extraire ~25 champs utiles
3. Sauvegarder un parquet intermédiaire (rechargement rapide)
4. Faire l'EDA sur ce parquet

## 0. Setup

In [3]:
import json
import re
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

BASE = Path('../data/otrf_datasets/datasets/compound/apt29')
ZIP_D1 = BASE / 'day1' / 'apt29_evals_day1_manual.zip'
ZIP_D2 = BASE / 'day2' / 'apt29_evals_day2_manual.zip'

OUT = Path('../results/eda')
OUT.mkdir(parents=True, exist_ok=True)
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f'Day 1 ZIP existe : {ZIP_D1.exists()} ({ZIP_D1.stat().st_size / 1e6:.0f} MB)')
print(f'Day 2 ZIP existe : {ZIP_D2.exists()} ({ZIP_D2.stat().st_size / 1e6:.0f} MB)')

Day 1 ZIP existe : True (14 MB)
Day 2 ZIP existe : True (43 MB)


## 1. Streaming + extraction des champs utiles

On garde uniquement les ~25 champs nécessaires pour la modélisation, parmi les 388 clés disponibles.

In [4]:
ESSENTIAL_FIELDS = [
    '@timestamp', 'EventID', 'Channel', 'Hostname', 'Message',
    # Sysmon process
    'Image', 'CommandLine', 'ParentImage', 'ParentCommandLine', 'User',
    'Hashes', 'Signature', 'Signed', 'IntegrityLevel',
    # Sysmon process access / image loaded
    'SourceImage', 'TargetImage', 'GrantedAccess', 'ImageLoaded',
    # Sysmon registry
    'TargetObject', 'EventType', 'Details',
    # Sysmon file
    'TargetFilename',
    # Sysmon network
    'DestinationIp', 'DestinationPort', 'Protocol', 'SourceIp',
    # Security / PowerShell
    'LogonType', 'AccountName', 'ScriptBlockText', 'Path',
]

def parse_event(line: bytes):
    """Parse une ligne JSON et garde uniquement les champs essentiels."""
    try:
        ev = json.loads(line)
    except json.JSONDecodeError:
        return None
    return {k: ev.get(k) for k in ESSENTIAL_FIELDS}

def stream_zip(zip_path: Path, day_label: str, limit: int = None):
    """Yield events from a ZIP file (streaming)."""
    with zipfile.ZipFile(zip_path) as z:
        # Le ZIP contient un seul fichier JSON
        name = z.namelist()[0]
        with z.open(name) as f:
            for i, line in enumerate(f):
                if limit and i >= limit:
                    break
                ev = parse_event(line)
                if ev:
                    ev['day'] = day_label
                    yield ev

print('Fonctions définies.')

Fonctions définies.


In [ ]:
# Streaming des 2 jours
import time

PARQUET = PROCESSED / 'events.parquet'

if PARQUET.exists():
    print(f'Parquet déjà existant — rechargement direct')
    df = pd.read_parquet(PARQUET)
else:
    print('Streaming Day 1...')
    t0 = time.time()
    rows_d1 = list(stream_zip(ZIP_D1, 'day1'))
    print(f'  -> {len(rows_d1):,} events en {time.time()-t0:.1f}s')

    print('Streaming Day 2 (1.6 GB, plus long)...')
    t0 = time.time()
    rows_d2 = list(stream_zip(ZIP_D2, 'day2'))
    print(f'  -> {len(rows_d2):,} events en {time.time()-t0:.1f}s')

    df = pd.DataFrame(rows_d1 + rows_d2)
    print(f'\nDataFrame combiné : {len(df):,} events, {df.shape[1]} colonnes')
    print(f'Mémoire : {df.memory_usage(deep=True).sum() / 1e6:.0f} MB')

    # Sauvegarde parquet (rechargement rapide ensuite)
    df.to_parquet(PARQUET, compression='snappy')
    print(f'Parquet sauvegardé : {PARQUET} ({PARQUET.stat().st_size / 1e6:.0f} MB)')

df.head(2)

Streaming Day 1...
  -> 196,081 events en 9.9s
Streaming Day 2 (1.6 GB, plus long)...


## 2. Nettoyage rapide (fusion casse Channel)

In [ ]:
# Fusion 'Security' / 'security'
df['Channel'] = df['Channel'].str.replace('^security$', 'Security', regex=True)
print('Channels après fusion casse :')
print(df['Channel'].value_counts().head(10).to_string())

## 3. Distribution globale Day 1 vs Day 2

In [ ]:
print('=== Volume par jour ===')
print(df['day'].value_counts().to_string())
print('\n=== Volume par Hostname ===')
print(df.groupby(['day', 'Hostname']).size().to_string())

print('\n=== Top 10 EventID par jour ===')
top_eid = df.groupby(['day', 'EventID']).size().sort_values(ascending=False).head(20)
print(top_eid.to_string())

In [ ]:
# Bar chart Channels par jour
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, day in zip(axes, ['day1', 'day2']):
    counts = df[df.day == day]['Channel'].value_counts().head(8)
    counts.plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'{day.upper()} — Top 8 Channels (total {df[df.day==day].shape[0]:,} events)')
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUT / 'channels_by_day.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap EventID Sysmon × Hostname
sysmon = df[df.Channel.str.contains('Sysmon', na=False)]
ct = pd.crosstab(sysmon['EventID'], sysmon['Hostname'])
top_eids = ct.sum(axis=1).sort_values(ascending=False).head(12).index
ct_top = ct.loc[top_eids]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(np.log1p(ct_top), annot=ct_top, fmt='d', cmap='Blues',
            cbar_kws={'label': 'log(count+1)'}, ax=ax)
ax.set_title('Sysmon EventID × Hostname (top 12 EventIDs)')
plt.tight_layout()
plt.savefig(OUT / 'eventid_x_hostname.png', dpi=110, bbox_inches='tight')
plt.show()

## 4. Timeline — frise temporelle des events

In [ ]:
df['ts'] = pd.to_datetime(df['@timestamp'], errors='coerce', utc=True)
df['hour'] = df['ts'].dt.hour
df['minute_bucket'] = df['ts'].dt.floor('5min')

# Events / 5min pour day1
for day in ['day1', 'day2']:
    sub = df[df.day == day]
    timeline = sub.groupby('minute_bucket').size()
    fig, ax = plt.subplots(figsize=(14, 3.5))
    timeline.plot(ax=ax, color='crimson', lw=1)
    ax.set_title(f'{day.upper()} — Volume d\'events par fenêtre de 5 min')
    ax.set_ylabel('Events / 5min')
    ax.set_xlabel('Temps')
    plt.tight_layout()
    plt.savefig(OUT / f'timeline_{day}.png', dpi=110, bbox_inches='tight')
    plt.show()

## 5. Analyse des PowerShell suspects

On regarde les `CommandLine` qui contiennent des patterns suspects.

In [ ]:
# Sysmon ProcessCreate (EventID 1) avec CommandLine non vide
proc = df[(df.EventID == 1) & (df.CommandLine.notna())].copy()
print(f'Sysmon ProcessCreate avec CommandLine : {len(proc):,}')

# Patterns suspects (extension MITRE T1059.001 PowerShell + autres)
patterns = {
    'powershell_encoded': r'-[eE][nNcC]',
    'powershell_hidden': r'-(?:[wW][iI][nNdD]|[hH][iI][dD])',
    'powershell_iex': r'IEX|Invoke-Expression',
    'powershell_downloadstring': r'DownloadString|DownloadFile|WebClient',
    'powershell_bypass': r'-[eE][xX][eE][cC].*[bB][yY][pP][aA][sS][sS]',
    'mimikatz_keyword': r'mimikatz|sekurlsa|lsadump',
    'wmic_exec': r'wmic.*(?:process|/node)',
    'schtasks': r'schtasks',
    'rundll32': r'rundll32',
    'regsvr32': r'regsvr32',
    'certutil_download': r'certutil.*-urlcache',
    'net_user_add': r'net\s+user.*\/add',
}

results = {}
for label, pat in patterns.items():
    mask = proc['CommandLine'].str.contains(pat, regex=True, na=False)
    n = int(mask.sum())
    results[label] = n
    print(f'  {label:30} : {n:>6} matches')

print(f'\nTotal events ProcessCreate suspects (au moins 1 pattern) :')
any_suspect = proc['CommandLine'].str.contains('|'.join(patterns.values()), regex=True, na=False)
print(f'  {any_suspect.sum():,} sur {len(proc):,} ({any_suspect.sum()/len(proc)*100:.1f}%)')

In [ ]:
# Visualisation : nombre de matches par pattern
ps_df = pd.DataFrame(list(results.items()), columns=['pattern', 'count']).sort_values('count', ascending=True)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(ps_df['pattern'], ps_df['count'], color='crimson')
ax.set_title('Patterns suspects détectés dans CommandLine (Sysmon EventID 1)')
ax.set_xlabel('Nombre de matches')
plt.tight_layout()
plt.savefig(OUT / 'suspicious_patterns.png', dpi=110, bbox_inches='tight')
plt.show()

## 6. Étiquetage prototype MITRE (test des règles)

On applique 10 règles MITRE simples pour estimer combien d'events seront étiquetés malveillants.

In [ ]:
def label_event(row):
    """Retourne (label, technique_mitre) — version prototype."""
    eid = row.get('EventID')
    
    # T1059.001 — PowerShell suspect (encoded, IEX, DownloadString...)
    if eid == 1:
        cmd = (row.get('CommandLine') or '').lower()
        if any(p in cmd for p in ['-enc ', '-encodedcommand', 'iex (', 'invoke-expression',
                                    'downloadstring', 'downloadfile', 'webclient']):
            return 1, 'T1059.001'
        if 'mimikatz' in cmd or 'sekurlsa' in cmd or 'lsadump' in cmd:
            return 1, 'T1003'
        if 'schtasks' in cmd and '/create' in cmd:
            return 1, 'T1053.005'
        if 'wmic' in cmd and ('process call create' in cmd or '/node:' in cmd):
            return 1, 'T1047'
        if 'certutil' in cmd and '-urlcache' in cmd:
            return 1, 'T1105'
        if re.search(r'net\s+user\s+\S+\s+\S+\s+/add', cmd):
            return 1, 'T1136.001'
    
    # T1003.001 — LSASS access
    if eid == 10:
        target = (row.get('TargetImage') or '').lower()
        granted = row.get('GrantedAccess', '') or ''
        if 'lsass.exe' in target:
            try:
                g = int(granted, 16) if isinstance(granted, str) and granted.startswith('0x') else int(granted)
                if g & 0x1010 or g & 0x0010:  # PROCESS_VM_READ
                    return 1, 'T1003.001'
            except (ValueError, TypeError):
                if 'lsass.exe' in target:
                    return 1, 'T1003.001'
    
    # T1547.001 — Registry Run Keys persistence
    if eid == 13:
        obj = (row.get('TargetObject') or '').lower()
        if r'\run' in obj or r'\runonce' in obj or r'\winlogon' in obj:
            return 1, 'T1547.001'
    
    # T1059.001 — PowerShell ScriptBlock suspect
    if eid == 4104:
        sb = (row.get('ScriptBlockText') or '').lower()
        if any(p in sb for p in ['invoke-mimikatz', 'invoke-bloodhound', 'invoke-kerberoast',
                                  'reflection.assembly', 'system.net.webclient',
                                  '$([char]', 'fromBase64String'.lower()]):
            return 1, 'T1059.001'
    
    return 0, None

# Apply (peut prendre 30s-1min sur 783k events)
import time
t0 = time.time()
labels = df.apply(label_event, axis=1, result_type='expand')
labels.columns = ['label', 'technique']
df['label'] = labels['label']
df['technique'] = labels['technique']
print(f'Étiquetage terminé en {time.time()-t0:.1f}s')
print(f'\nLabel distribution :')
print(df['label'].value_counts().to_string())
print(f'\nLabel rate : {df["label"].mean()*100:.2f}%')

In [ ]:
# Détail par technique MITRE
print('=== Events étiquetés par technique MITRE ===')
by_tech = df[df.label == 1]['technique'].value_counts()
print(by_tech.to_string())

# Par jour
print('\n=== Events étiquetés par jour ===')
print(df[df.label == 1].groupby('day').size().to_string())

# Par hôte
print('\n=== Events étiquetés par hôte ===')
print(df[df.label == 1]['Hostname'].value_counts().to_string())

In [ ]:
# Visualisation : étiquetage par technique
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

by_tech.plot.bar(ax=axes[0], color='crimson')
axes[0].set_title(f'Events étiquetés MALVEILLANT par technique MITRE\n(total {df["label"].sum():,} = {df["label"].mean()*100:.2f}%)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Nombre d\'events')

df.groupby(['day', 'label']).size().unstack(fill_value=0).plot.bar(ax=axes[1], stacked=False, color=['steelblue', 'crimson'])
axes[1].set_title('Répartition Normal vs Malveillant par jour')
axes[1].set_xticklabels(['Day 1', 'Day 2'], rotation=0)
axes[1].legend(['Normal (0)', 'Malveillant (1)'])

plt.tight_layout()
plt.savefig(OUT / 'labeling_prototype.png', dpi=110, bbox_inches='tight')
plt.show()

## 7. Exemples concrets d'events étiquetés malveillants (pour validation manuelle)

In [ ]:
print('=== Exemples par technique MITRE ===\n')
for tech in df[df.label == 1]['technique'].unique():
    examples = df[(df.label == 1) & (df.technique == tech)].head(2)
    print(f'### {tech} — {len(df[(df.label==1)&(df.technique==tech)])} events')
    for _, row in examples.iterrows():
        cmd = row.get('CommandLine') or row.get('TargetObject') or row.get('TargetImage') or row.get('ScriptBlockText') or ''
        cmd_short = str(cmd)[:200]
        print(f'  [{row.day}] EventID={row.EventID} Host={row.Hostname[:15]:15s} | {cmd_short}')
    print()

## 8. Anomalies & qualité des données

In [ ]:
print('=== Couverture des champs essentiels ===')
coverage = (df.notna().sum() / len(df) * 100).round(1).sort_values(ascending=False)
print(coverage.to_string())

print('\n=== Doublons exacts (sur tous les champs essentiels) ===')
n_dup = df.drop(columns=['ts','hour','minute_bucket','label','technique'], errors='ignore').duplicated().sum()
print(f'Doublons : {n_dup:,} ({n_dup/len(df)*100:.2f}%)')

## 9. Sauvegarde du résumé EDA

In [ ]:
summary = {
    'total_events': int(len(df)),
    'day1_events': int((df.day == 'day1').sum()),
    'day2_events': int((df.day == 'day2').sum()),
    'channels': df['Channel'].value_counts().head(10).to_dict(),
    'top_eventid_sysmon': sysmon['EventID'].value_counts().head(10).to_dict(),
    'hostnames': df['Hostname'].value_counts().to_dict(),
    'duplicates_essential': int(n_dup),
    'duplicates_pct': float(n_dup/len(df)*100),
    'field_coverage_pct': coverage.to_dict(),
    'suspicious_patterns_processcreate': results,
    'prototype_labeling': {
        'total_labeled_malicious': int(df['label'].sum()),
        'label_rate_pct': float(df['label'].mean() * 100),
        'by_technique': by_tech.to_dict(),
        'by_day': df[df.label==1].groupby('day').size().to_dict(),
        'by_hostname': df[df.label==1]['Hostname'].value_counts().to_dict(),
    },
    'parquet_path': str(PARQUET),
    'parquet_size_mb': round(PARQUET.stat().st_size / 1e6, 1),
}
import json as json_lib
(OUT / 'eda_summary.json').write_text(json_lib.dumps(summary, indent=2, default=str), encoding='utf-8')
print('Résumé EDA sauvegardé :', OUT / 'eda_summary.json')
print()
print(json_lib.dumps({k:v for k,v in summary.items() if not isinstance(v,dict) or len(str(v))<500}, indent=2, default=str))

## 10. Conclusions Phase 1

À remplir après exécution (cellule markdown éditable).

**Métriques clés à reporter :**
- Volume total : ?
- Doublons : ?
- Taux d'étiquetage malveillant : ?
- Techniques MITRE couvertes : ?
- Décisions Phase 2 : ?